# 서울 국민평형 아파트 실거래가 분석

미니 프로젝트 1 / Python · SQL

2020년부터 2026년 11월 20일까지의 서울 아파트 매매 실거래가를 분석했습니다. 전용면적 83~85㎡ 거래를 추출하고, 계약 해제 건을 제외한 뒤 MariaDB에 저장했습니다. 이후 Python과 SQL로 거래량과 가격을 집계했습니다.

- 데이터: 국토교통부 실거래가 공개시스템 (2020~2026년 CSV 7개)
- 대상: 서울 25개 자치구, 전용면적 83~85㎡
- 도구: Python, pandas, MariaDB, SQL

## 1. CSV 불러오기

연도별 CSV 파일을 불러와 하나로 합칩니다. 원본 파일은 `raw` 폴더에 저장했습니다.

In [31]:
# 데이터 파일 목록 확인

from pathlib import Path

# CSV 파일이 저장된 폴더 경로 설정
data_dir = Path.home() / "Documents" / "과제" / "realestate_project" / "raw"

# CSV 파일 목록 가져오기
csv_files = sorted(data_dir.glob("*.csv"))

# 파일 개수 출력
print("CSV 파일 개수:", len(csv_files))

# 파일 이름 출력
for file in csv_files:
    print(file.name)

CSV 파일 개수: 19
(월) 매매수급동향_아파트.csv
2020-26_구글트랜드_부동산.csv
2020아파트(매매)_실거래가.csv
2021아파트(매매)_실거래가.csv
2022아파트(매매)_실거래가.csv
2023아파트(매매)_실거래가.csv
2024아파트(매매)_실거래가.csv
2025아파트(매매)_실거래가.csv
2026_구글트랜드_아파트매매 261120까지.csv
2026_구글트랜드_집값 261120까지.csv
2026아파트(매매)_실거래가261120까지.csv
capital_region_mortgage_2020_2026.csv
m2_growth_2020_2026.csv
seoul_apartment_jeonse_index_2020_2026.csv
seoul_apartment_sales_supply_index_2020_2026.csv
seoul_apartment_sales_volume_2020_2026.csv
seoul_housing_market_indicators_2020_2026.csv
seoul_housing_sales_sentiment_2020_2026.csv
주택매매시장_소비심리지수20-26.csv


In [32]:
# 2020년 아파트 매매 데이터 구조 확인

# 2020년 CSV 파일 선택
apt_file = [f for f in csv_files if f.name.startswith("2020아파트")][0]

# 파일 내용 앞부분 확인
with open(apt_file, "r", encoding="cp949") as f:
    for i in range(20):
        print(f.readline().strip())

"□ 본 서비스에서 제공하는 정보는 법적인 효력이 없으므로 참고용으로만 활용하시기 바랍니다."
"□ 신고정보가 실시간 변경, 해제되어 제공시점에 따라 공개건수 및 내용이 상이할 수 있는 점 참고하시기 바랍니다."
"□ 본 자료는 계약일 기준입니다. (※ 7월 계약, 8월 신고건 → 7월 거래건으로  제공)"
"□ 통계자료 활용시에는 수치가 왜곡될 수 있으니 참고자료로만 활용하시기  바라며,  외부 공개시에는 반드시 신고일 기준으로 집계되는 공식통계를 이용하여 주시기 바랍니다."
""
"* 국토교통부 실거래가 공개시스템의 궁금하신 점이나 문의사항은 콜센터 1533-2949로 연락 주시기 바랍니다."
"□ 검색조건"
"계약일자 : 2020-01-01 ~ 2020-12-31"
"실거래구분 : 아파트(매매)"
"주소구분 : 지번주소"
"시도 : 서울특별시"
"시군구 : 전체"
"읍면동 : 전체"
"면적 : 60㎡초과~85㎡이하"
"금액선택 : 전체"
"NO","시군구","번지","본번","부번","단지명","전용면적(㎡)","계약년월","계약일","거래금액(만원)","동","층","매수자","매도자","건축년도","도로명","해제사유발생일","거래유형","중개사소재지","등기일자"
"1","서울특별시 성북구 정릉동","1016","1016","0000","우방","84.9200","202012","31","54,000","-","12","-","-","1999","북악산로 746","-","-","-","-"
"2","서울특별시 성북구 길음동","1279","1279","0000","길음뉴타운1단지(래미안길음1차)","84.9600","202012","31","105,000","-","16","-","-","2003","길음로9길 40","-","-","-","-"
"3","서울특별시 서대문구 홍제동","82","0082","0000","홍제한양","60.0600","202012","31","71,000","-","3","-","-","1993","통일로25길 30"

In [33]:
# 아파트 실거래가 파일별 조회 기간 확인

apt_files = sorted(
    f for f in csv_files if "아파트" in f.name and "실거래가" in f.name
)

for file in apt_files:

    with open(file, "r", encoding="cp949") as f:

        for line in f:

            if "계약일자 :" in line:
                print(file.name)
                print(line.strip())
                print()
                break

2020아파트(매매)_실거래가.csv
"계약일자 : 2020-01-01 ~ 2020-12-31"

2021아파트(매매)_실거래가.csv
"계약일자 : 2021-01-01 ~ 2021-12-31"

2022아파트(매매)_실거래가.csv
"계약일자 : 2022-01-01 ~ 2022-12-31"

2023아파트(매매)_실거래가.csv
"계약일자 : 2023-01-01 ~ 2023-12-31"

2024아파트(매매)_실거래가.csv
"계약일자 : 2024-01-01 ~ 2024-12-31"

2025아파트(매매)_실거래가.csv
"계약일자 : 2025-01-01 ~ 2025-12-31"

2026아파트(매매)_실거래가261120까지.csv
"계약일자 : 2026-01-01 ~ 2026-11-20"



In [34]:
# 2020년 아파트 실거래가 데이터 불러오기

import pandas as pd

# 2020년 파일 선택
apt_file = data_dir / "2020아파트(매매)_실거래가.csv"

# 실제 컬럼명이 있는 줄 찾기
with open(apt_file, "r", encoding="cp949") as f:
    for i, line in enumerate(f):
        if '"NO"' in line and "전용면적" in line:
            header_row = i
            break

# CSV 불러오기
df_2020 = pd.read_csv(
    apt_file,
    encoding="cp949",
    skiprows=header_row
)

# 데이터 크기 확인
print("데이터 크기:", df_2020.shape)

# 상위 5개 행 출력
display(df_2020.head())

데이터 크기: (34620, 20)


,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자
0,1,서울특별시 성북구 정릉동,1016,1016,0,우방,84.92,202012,31,"54,000",-,12,-,-,1999,북악산로 746,-,-,-,-
1,2,서울특별시 성북구 길음동,1279,1279,0,길음뉴타운1단지(래미안길음1차),84.96,202012,31,"105,000",-,16,-,-,2003,길음로9길 40,-,-,-,-
2,3,서울특별시 서대문구 홍제동,82,82,0,홍제한양,60.06,202012,31,"71,000",-,3,-,-,1993,통일로25길 30,-,-,-,-
3,4,서울특별시 서대문구 홍제동,82,82,0,홍제한양,60.06,202012,31,"73,000",-,11,-,-,1993,통일로25길 30,-,-,-,-
4,5,서울특별시 서대문구 남가좌동,390,390,0,래미안루센티아,84.95,202012,31,"127,500",-,12,-,-,2020,거북골로 100,-,-,-,-


In [35]:
# 2020~2026년 아파트 매매 데이터 통합

import pandas as pd

# 아파트 CSV 파일 목록
apt_files = sorted(data_dir.glob("*아파트(매매)_실거래가*.csv"))

# 데이터를 저장할 빈 리스트
df_list = []

# CSV 파일을 하나씩 불러오기
for file in apt_files:

    # 실제 컬럼명이 있는 줄 찾기
    with open(file, "r", encoding="cp949") as f:
        for i, line in enumerate(f):
            if '"NO"' in line and "전용면적" in line:
                header_row = i
                break

    # CSV 불러오기
    df = pd.read_csv(
        file,
        encoding="cp949",
        skiprows=header_row,
        low_memory=False
    )

    # 원본 파일 이름 기록
    df["원본파일"] = file.name

    # 리스트에 추가
    df_list.append(df)

    print(file.name, ":", len(df), "건")

# 7개 데이터프레임 합치기
apt_all = pd.concat(df_list, ignore_index=True)

# 전체 데이터 크기 확인
print("\n통합 데이터 크기:", apt_all.shape)

# 통합 데이터 상위 5개 확인
display(apt_all.head())

2020아파트(매매)_실거래가.csv : 34620 건
2021아파트(매매)_실거래가.csv : 16465 건
2022아파트(매매)_실거래가.csv : 4043 건
2023아파트(매매)_실거래가.csv : 14129 건
2024아파트(매매)_실거래가.csv : 24285 건
2025아파트(매매)_실거래가.csv : 35586 건
2026아파트(매매)_실거래가261120까지.csv : 24300 건

통합 데이터 크기: (153428, 21)


,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,원본파일
0,1,서울특별시 성북구 정릉동,1016,1016,0,우방,84.92,202012,31,"54,000",...,12,-,-,1999,북악산로 746,-,-,-,-,2020아파트(매매)_실거래가.csv
1,2,서울특별시 성북구 길음동,1279,1279,0,길음뉴타운1단지(래미안길음1차),84.96,202012,31,"105,000",...,16,-,-,2003,길음로9길 40,-,-,-,-,2020아파트(매매)_실거래가.csv
2,3,서울특별시 서대문구 홍제동,82,82,0,홍제한양,60.06,202012,31,"71,000",...,3,-,-,1993,통일로25길 30,-,-,-,-,2020아파트(매매)_실거래가.csv
3,4,서울특별시 서대문구 홍제동,82,82,0,홍제한양,60.06,202012,31,"73,000",...,11,-,-,1993,통일로25길 30,-,-,-,-,2020아파트(매매)_실거래가.csv
4,5,서울특별시 서대문구 남가좌동,390,390,0,래미안루센티아,84.95,202012,31,"127,500",...,12,-,-,2020,거북골로 100,-,-,-,-,2020아파트(매매)_실거래가.csv


## 2. 데이터 전처리

전용면적 83~85㎡ 거래만 남기고 거래금액을 숫자로 변환합니다. 계약이 해제된 거래는 분석 대상에서 제외합니다.

In [36]:
# 국민평형 아파트 데이터 추출

# 전용면적을 숫자형으로 변환
apt_all["전용면적(㎡)"] = pd.to_numeric(
    apt_all["전용면적(㎡)"],
    errors="coerce"
)

# 전용면적 83~85㎡ 필터링
apt_84 = apt_all[
    (apt_all["전용면적(㎡)"] >= 83) &
    (apt_all["전용면적(㎡)"] <= 85)
].copy()

# 계약년월에서 연도 추출
apt_84["거래연도"] = (
    pd.to_numeric(apt_84["계약년월"], errors="coerce") // 100
).astype("Int64")

# 필터링 결과 확인
print("전체 거래 건수:", len(apt_all))
print("국민평형 거래 건수:", len(apt_84))
print("국민평형 비율:", round(len(apt_84) / len(apt_all) * 100, 2), "%")

# 연도별 거래 건수
print("\n연도별 국민평형 거래 건수")
print(apt_84.groupby("거래연도").size())

# 상위 5개 데이터 확인
display(apt_84.head())

전체 거래 건수: 153428
국민평형 거래 건수: 118707
국민평형 비율: 77.37 %

연도별 국민평형 거래 건수
거래연도
2020    26062
2021    12327
2022     2917
2023    11263
2024    19347
2025    28261
2026    18530
dtype: int64


,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,원본파일,거래연도
0,1,서울특별시 성북구 정릉동,1016,1016,0,우방,84.92,202012,31,"54,000",...,-,-,1999,북악산로 746,-,-,-,-,2020아파트(매매)_실거래가.csv,2020
1,2,서울특별시 성북구 길음동,1279,1279,0,길음뉴타운1단지(래미안길음1차),84.96,202012,31,"105,000",...,-,-,2003,길음로9길 40,-,-,-,-,2020아파트(매매)_실거래가.csv,2020
4,5,서울특별시 서대문구 남가좌동,390,390,0,래미안루센티아,84.95,202012,31,"127,500",...,-,-,2020,거북골로 100,-,-,-,-,2020아파트(매매)_실거래가.csv,2020
5,6,서울특별시 성동구 하왕십리동,1066,1066,0,텐즈힐(1단지),84.88,202012,31,"161,500",...,-,-,2015,무학로 33,-,-,-,-,2020아파트(매매)_실거래가.csv,2020
6,7,서울특별시 동대문구 이문동,54,54,0,현대,84.56,202012,31,"83,000",...,-,-,2000,한천로58길 107,-,-,-,-,2020아파트(매매)_실거래가.csv,2020


In [37]:
# 거래금액 데이터 타입 변환

# 거래금액에서 쉼표 제거 후 숫자형 변환
apt_84["거래금액_만원"] = pd.to_numeric(
    apt_84["거래금액(만원)"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

# 데이터 타입 확인
print("거래금액 데이터 타입:")
print(apt_84["거래금액_만원"].dtype)

# 결측치 확인
print("\n거래금액 결측치:", apt_84["거래금액_만원"].isna().sum())

# 거래금액 기초 통계
print("\n거래금액 기초 통계 (단위: 만원)")
print(apt_84["거래금액_만원"].describe())

# 변환 결과 확인
display(apt_84[[
    "시군구",
    "단지명",
    "전용면적(㎡)",
    "거래금액(만원)",
    "거래금액_만원"
]].head())

거래금액 데이터 타입:
int64

거래금액 결측치: 0

거래금액 기초 통계 (단위: 만원)
count    118707.000000
mean     120055.748431
std       63548.261645
min       16000.000000
25%       77000.000000
50%      104000.000000
75%      145000.000000
max      720000.000000
Name: 거래금액_만원, dtype: float64


,시군구,단지명,전용면적(㎡),거래금액(만원),거래금액_만원
0,서울특별시 성북구 정릉동,우방,84.92,"54,000",54000
1,서울특별시 성북구 길음동,길음뉴타운1단지(래미안길음1차),84.96,"105,000",105000
4,서울특별시 서대문구 남가좌동,래미안루센티아,84.95,"127,500",127500
5,서울특별시 성동구 하왕십리동,텐즈힐(1단지),84.88,"161,500",161500
6,서울특별시 동대문구 이문동,현대,84.56,"83,000",83000


In [38]:
# 주요 컬럼의 결측치 및 계약 해제 거래 확인

# 확인할 컬럼 목록
check_cols = [
    "시군구",
    "단지명",
    "전용면적(㎡)",
    "계약년월",
    "계약일",
    "거래금액_만원",
    "층",
    "건축년도",
    "해제사유발생일"
]

# 실제 결측치 확인
print("1. 컬럼별 결측치")
print(apt_84[check_cols].isna().sum())

# '-'로 표시된 값 확인
print("\n2. '-'로 표시된 값의 개수")

for col in check_cols:
    count = (apt_84[col].astype(str).str.strip() == "-").sum()
    print(col, ":", count)

# 계약 해제 정보 확인
print("\n3. 계약 해제 정보")
print(apt_84["해제사유발생일"].value_counts(dropna=False).head(10))

1. 컬럼별 결측치
시군구        0
단지명        0
전용면적(㎡)    0
계약년월       0
계약일        0
거래금액_만원    0
층          0
건축년도       0
해제사유발생일    0
dtype: int64

2. '-'로 표시된 값의 개수
시군구 : 0
단지명 : 0
전용면적(㎡) : 0
계약년월 : 0
계약일 : 0
거래금액_만원 : 0
층 : 0
건축년도 : 0
해제사유발생일 : 113083

3. 계약 해제 정보
해제사유발생일
-           113083
20250627        43
20250624        37
20250626        29
20250620        27
20250623        26
20250604        26
20250625        25
20250619        24
20250616        21
Name: count, dtype: int64


In [39]:
# 계약 해제 거래 제거

# 계약 해제 여부 확인
cancel_date = apt_84["해제사유발생일"].astype(str).str.strip()

# 정상 거래와 계약 해제 거래 구분
normal = cancel_date == "-"
cancelled = cancel_date.str.fullmatch(r"\d{8}")

# 예상하지 못한 값 확인
other = ~(normal | cancelled)

print("전체 거래 건수:", len(apt_84))
print("계약 해제일 미기록:", normal.sum())
print("계약 해제 거래:", cancelled.sum())
print("기타 값:", other.sum())

# 기타 값이 없는 경우 계약 해제 거래 제외
if other.sum() == 0:
    apt_clean = apt_84[normal].copy()

    print("\n정제 후 거래 건수:", len(apt_clean))
else:
    print("\n기타 값을 먼저 확인해야 합니다.")
    print(cancel_date[other].value_counts())

전체 거래 건수: 118707
계약 해제일 미기록: 113083
계약 해제 거래: 5624
기타 값: 0

정제 후 거래 건수: 113083


## 3. 중복 및 이상치 확인

중복으로 보이는 거래와 가격 이상치를 확인합니다. 실제 거래일 가능성이 있어 이 단계에서는 삭제하지 않았습니다.

In [40]:
# 중복 거래 검사

# 중복 여부를 판단할 주요 컬럼
duplicate_cols = [
    "시군구",
    "단지명",
    "전용면적(㎡)",
    "계약년월",
    "계약일",
    "거래금액_만원",
    "층"
]

# 중복 의심 거래 찾기
duplicates = apt_clean[
    apt_clean.duplicated(
        subset=duplicate_cols,
        keep=False
    )
].copy()

# 결과 출력
print("전체 거래 건수:", len(apt_clean))
print("중복 의심 거래 건수:", len(duplicates))

# 중복 의심 거래 확인
display(
    duplicates[duplicate_cols]
    .sort_values(duplicate_cols)
    .head(20)
)

전체 거래 건수: 113083
중복 의심 거래 건수: 509


,시군구,단지명,전용면적(㎡),계약년월,계약일,거래금액_만원,층
75835,서울특별시 강남구 개포동,개포자이프레지던스,84.5978,202408,21,320000,5
75850,서울특별시 강남구 개포동,개포자이프레지던스,84.5978,202408,21,320000,5
37899,서울특별시 강남구 개포동,아람손프라자/비바빌,84.9600,202108,25,77000,5
37902,서울특별시 강남구 개포동,아람손프라자/비바빌,84.9600,202108,25,77000,5
97126,서울특별시 강남구 대치동,대치삼성,84.5800,202510,31,347000,2
97127,서울특별시 강남구 대치동,대치삼성,84.5800,202510,31,347000,2
129555,서울특별시 강남구 대치동,은마,84.4300,202610,31,350000,14
129559,서울특별시 강남구 대치동,은마,84.4300,202610,31,350000,14
79923,서울특별시 강남구 도곡동,도곡렉슬,84.9231,202407,14,310000,15
79924,서울특별시 강남구 도곡동,도곡렉슬,84.9231,202407,14,310000,15


In [41]:
# 중복 거래 상세 검사

# 비교에서 제외할 컬럼
exclude_cols = [
    "NO",
    "원본파일",
    "거래연도",
    "거래금액_만원"
]

# 나머지 컬럼으로 비교
compare_cols = [
    col for col in apt_clean.columns
    if col not in exclude_cols
]

# 상세 정보까지 동일한 거래 찾기
exact_duplicates = apt_clean[
    apt_clean.duplicated(
        subset=compare_cols,
        keep=False
    )
].copy()

# 검사 결과
print("전체 거래 건수:", len(apt_clean))
print("기본 중복 의심 거래:", len(duplicates))
print("상세 정보까지 동일한 거래:", len(exact_duplicates))

# 중복 의심 거래의 추가 정보 확인
detail_cols = duplicate_cols + [
    "NO",
    "본번",
    "부번",
    "건축년도",
    "거래유형",
    "등기일자",
    "원본파일"
]

display(
    duplicates[detail_cols]
    .sort_values(duplicate_cols)
    .head(20)
)

전체 거래 건수: 113083
기본 중복 의심 거래: 509
상세 정보까지 동일한 거래: 349


,시군구,단지명,전용면적(㎡),계약년월,계약일,거래금액_만원,층,NO,본번,부번,건축년도,거래유형,등기일자,원본파일
75835,서울특별시 강남구 개포동,개포자이프레지던스,84.5978,202408,21,320000,5,6579,1284,0,2023,중개거래,25.05.23,2024아파트(매매)_실거래가.csv
75850,서울특별시 강남구 개포동,개포자이프레지던스,84.5978,202408,21,320000,5,6594,1284,0,2023,중개거래,25.05.23,2024아파트(매매)_실거래가.csv
37899,서울특별시 강남구 개포동,아람손프라자/비바빌,84.9600,202108,25,77000,5,3280,1164,7,2001,-,-,2021아파트(매매)_실거래가.csv
37902,서울특별시 강남구 개포동,아람손프라자/비바빌,84.9600,202108,25,77000,5,3283,1164,7,2001,-,-,2021아파트(매매)_실거래가.csv
97126,서울특별시 강남구 대치동,대치삼성,84.5800,202510,31,347000,2,3585,1014,3,2000,중개거래,26.02.04,2025아파트(매매)_실거래가.csv
97127,서울특별시 강남구 대치동,대치삼성,84.5800,202510,31,347000,2,3586,1014,3,2000,중개거래,26.02.04,2025아파트(매매)_실거래가.csv
129555,서울특별시 강남구 대치동,은마,84.4300,202610,31,350000,14,428,316,0,1980,중개거래,-,2026아파트(매매)_실거래가261120까지.csv
129559,서울특별시 강남구 대치동,은마,84.4300,202610,31,350000,14,432,316,0,1980,중개거래,-,2026아파트(매매)_실거래가261120까지.csv
79923,서울특별시 강남구 도곡동,도곡렉슬,84.9231,202407,14,310000,15,10667,527,0,2006,중개거래,25.04.07,2024아파트(매매)_실거래가.csv
79924,서울특별시 강남구 도곡동,도곡렉슬,84.9231,202407,14,310000,15,10668,527,0,2006,중개거래,-,2024아파트(매매)_실거래가.csv


In [42]:
# 거래금액 이상치 검사

# 거래금액 데이터
price = apt_clean["거래금액_만원"]

# 사분위수 계산
Q1 = price.quantile(0.25)
Q3 = price.quantile(0.75)

# IQR 계산
IQR = Q3 - Q1

# 이상치 판단 기준
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# 이상치 후보 추출
outliers = apt_clean[
    (price < lower) | (price > upper)
].copy()

# 결과 출력
print("전체 거래 건수:", len(apt_clean))
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("이상치 하한:", lower)
print("이상치 상한:", upper)
print("이상치 후보 건수:", len(outliers))

# 가격이 가장 높은 거래 10건
print("\n최고가 거래 10건")

display(
    apt_clean[
        ["시군구", "단지명", "계약년월", "거래금액_만원"]
    ].sort_values(
        "거래금액_만원",
        ascending=False
    ).head(10)
)

# 가격이 가장 낮은 거래 10건
print("\n최저가 거래 10건")

display(
    apt_clean[
        ["시군구", "단지명", "계약년월", "거래금액_만원"]
    ].sort_values(
        "거래금액_만원"
    ).head(10)
)

전체 거래 건수: 113083
Q1: 77000.0
Q3: 145000.0
IQR: 68000.0
이상치 하한: -25000.0
이상치 상한: 247000.0
이상치 후보 건수: 5420

최고가 거래 10건


,시군구,단지명,계약년월,거래금액_만원
109477,서울특별시 서초구 반포동,래미안원베일리,202506,720000
106043,서울특별시 서초구 반포동,래미안원베일리,202508,715000
139195,서울특별시 서초구 반포동,래미안원베일리,202605,715000
124391,서울특별시 서초구 반포동,래미안원베일리,202503,700000
124828,서울특별시 서초구 반포동,래미안원베일리,202502,665000
109480,서울특별시 서초구 반포동,래미안원베일리,202506,660000
141417,서울특별시 강남구 압구정동,"현대14차(203,204,205,206동)",202605,660000
99414,서울특별시 서초구 반포동,래미안원베일리,202510,651000
139717,서울특별시 서초구 반포동,래미안원베일리,202605,650000
108484,서울특별시 강남구 압구정동,"현대14차(203,204,205,206동)",202507,650000



최저가 거래 10건


,시군구,단지명,계약년월,거래금액_만원
48497,서울특별시 양천구 신월동,금하뜨라네,202102,16000
58634,서울특별시 도봉구 창동,신창빌라트,202309,19000
41020,서울특별시 강북구 미아동,인성,202107,20800
26124,서울특별시 은평구 구산동,타워,202004,22500
29977,서울특별시 은평구 신사동,명진(200-115),202002,22500
14231,서울특별시 금천구 시흥동,유승셀르빌,202007,22700
2746,서울특별시 도봉구 방학동,태양아파트(612-2),202012,23000
22113,서울특별시 은평구 역촌동,엔에스,202006,24500
26370,서울특별시 구로구 오류동,시티월드,202004,24800
13271,서울특별시 은평구 구산동,우공아트빌,202007,25000


In [43]:
# 계약월 및 자치구 컬럼 생성

# 계약년월을 날짜형으로 변환
apt_clean["계약월"] = pd.to_datetime(
    apt_clean["계약년월"].astype(str),
    format="%Y%m",
    errors="coerce"
)

# 시군구에서 자치구 이름 추출
apt_clean["자치구"] = (
    apt_clean["시군구"].str.split().str[1]
)

# 날짜 결측치 확인
print("계약월 결측치:", apt_clean["계약월"].isna().sum())

# 자치구 개수 확인
print("자치구 개수:", apt_clean["자치구"].nunique())

# 계약월 범위 확인
print("최초 계약월:", apt_clean["계약월"].min())
print("최종 계약월:", apt_clean["계약월"].max())

# 결과 확인
display(
    apt_clean[
        ["시군구", "자치구", "계약년월", "계약월", "거래금액_만원"]
    ].head()
)

계약월 결측치: 0
자치구 개수: 25
최초 계약월: 2020-01-01 00:00:00
최종 계약월: 2026-11-01 00:00:00


,시군구,자치구,계약년월,계약월,거래금액_만원
0,서울특별시 성북구 정릉동,성북구,202012,2020-12-01,54000
1,서울특별시 성북구 길음동,성북구,202012,2020-12-01,105000
4,서울특별시 서대문구 남가좌동,서대문구,202012,2020-12-01,127500
5,서울특별시 성동구 하왕십리동,성동구,202012,2020-12-01,161500
6,서울특별시 동대문구 이문동,동대문구,202012,2020-12-01,83000


## 4. Python 통계 집계

계약월별 거래량과 평균·중위 거래가격을 구하고, 2025년 자치구별 결과도 확인합니다.

In [44]:
# 월별 아파트 거래가격 통계 분석

# 계약월별 거래건수, 평균가격, 중위가격 계산
monthly_stats = apt_clean.groupby("계약월").agg(
    거래건수=("거래금액_만원", "count"),
    평균가격_만원=("거래금액_만원", "mean"),
    중위가격_만원=("거래금액_만원", "median")
).reset_index()

# 날짜순 정렬
monthly_stats = monthly_stats.sort_values(
    "계약월"
).reset_index(drop=True)

# 결과 확인
print("월별 데이터 개수:", len(monthly_stats))
print("전체 거래 건수:", monthly_stats["거래건수"].sum())
print("\n컬럼별 결측치")
print(monthly_stats.isna().sum())
print("\n처음 5개월")
display(monthly_stats.head())
print("\n마지막 5개월")
display(monthly_stats.tail())

월별 데이터 개수: 83
전체 거래 건수: 113083

컬럼별 결측치
계약월        0
거래건수       0
평균가격_만원    0
중위가격_만원    0
dtype: int64

처음 5개월


,계약월,거래건수,평균가격_만원,중위가격_만원
0,2020-01-01,2017,73440.934556,66200.0
1,2020-02-01,2616,77600.717125,69200.0
2,2020-03-01,1321,74904.355791,67000.0
3,2020-04-01,850,77526.717647,65500.0
4,2020-05-01,1650,88371.381818,75550.0



마지막 5개월


,계약월,거래건수,평균가격_만원,중위가격_만원
78,2026-07-01,1761,129996.080068,114700.0
79,2026-08-01,1124,122548.904804,105000.0
80,2026-09-01,1024,130380.029297,107375.0
81,2026-10-01,1221,136711.203931,119000.0
82,2026-11-01,296,120827.364865,105500.0


In [45]:
# 2025년 서울 자치구별 아파트 매매가격 분석

# 2025년 거래만 추출
apt_2025 = apt_clean[
    apt_clean["계약월"].dt.year == 2025
].copy()

# 자치구별 통계 계산
district_stats = apt_2025.groupby("자치구").agg(
    거래건수=("거래금액_만원", "count"),
    평균가격_만원=("거래금액_만원", "mean"),
    중위가격_만원=("거래금액_만원", "median")
).reset_index()

# 가격을 억 원 단위로 변환
district_stats["평균가격_억원"] = (
    district_stats["평균가격_만원"] / 10000
).round(2)

district_stats["중위가격_억원"] = (
    district_stats["중위가격_만원"] / 10000
).round(2)

# 중위가격 기준 내림차순 정렬
district_stats = district_stats.sort_values(
    "중위가격_만원",
    ascending=False
).reset_index(drop=True)

# 결과 확인
print("분석 연도: 2025년")
print("자치구 개수:", len(district_stats))
print("전체 거래 건수:", district_stats["거래건수"].sum())

display(
    district_stats[
        ["자치구", "거래건수", "평균가격_억원", "중위가격_억원"]
    ]
)

분석 연도: 2025년
자치구 개수: 25
전체 거래 건수: 26105


,자치구,거래건수,평균가격_억원,중위가격_억원
0,서초구,892,27.88,26.50
1,강남구,996,26.30,25.75
2,송파구,1899,19.92,18.40
3,용산구,341,18.98,17.50
4,성동구,1601,16.88,16.30
5,광진구,798,15.71,15.60
6,마포구,1529,16.13,15.35
7,강동구,1958,13.74,13.82
8,동작구,1779,14.16,13.70
9,종로구,208,15.54,13.46


## 5. MariaDB 저장

정제한 거래 데이터는 `realestate_project` 데이터베이스의 `apartment_sales` 테이블에 저장합니다. 이미 데이터가 있으면 다시 넣지 않도록 했습니다.

In [46]:
# MariaDB 연결 준비
import sys
import os
from getpass import getpass
from pathlib import Path

legacy_db_dir = Path.home() / "Documents" / "과제"
if not (Path.cwd() / "db.py").exists() and str(legacy_db_dir) not in sys.path:
    sys.path.append(str(legacy_db_dir))

from db import get_conn

# 비밀번호가 설정되지 않았다면 입력
if not os.environ.get("DB_PASSWORD"):
    os.environ["DB_PASSWORD"] = getpass("MariaDB 비밀번호: ")


In [47]:
# 부동산 프로젝트 데이터베이스 연결 확인

from db import get_conn

# 프로젝트 데이터베이스 연결
conn = get_conn("realestate_project")
cur = conn.cursor()

# 현재 데이터베이스 확인
cur.execute("SELECT DATABASE()")

print("현재 데이터베이스:", cur.fetchone()[0])
print("데이터베이스 연결 성공!")

conn.close()

현재 데이터베이스: realestate_project
데이터베이스 연결 성공!


In [48]:
# MariaDB 프로젝트 거래 테이블 생성

from db import get_conn

conn = get_conn("realestate_project")
cur = conn.cursor()

# 아파트 매매 데이터 테이블
cur.execute("""
CREATE TABLE IF NOT EXISTS apartment_sales (
    id INT AUTO_INCREMENT PRIMARY KEY,
    contract_month DATE,
    district VARCHAR(30),
    location VARCHAR(100),
    apartment_name VARCHAR(255),
    area DECIMAL(10,4),
    floor INT,
    build_year INT,
    price_manwon INT
)
""")

conn.commit()
print("테이블 생성 완료!")

# 생성된 테이블 확인
cur.execute("SHOW TABLES")
for row in cur.fetchall():
    print(row[0])

conn.close()

테이블 생성 완료!
apartment_sales
search_trends


In [49]:
# 아파트 실거래가 데이터를 MariaDB에 저장

import pandas as pd
from db import get_conn

# 1. 저장할 컬럼 선택
apt_sql = apt_clean[[
    "계약월",
    "자치구",
    "시군구",
    "단지명",
    "전용면적(㎡)",
    "층",
    "건축년도",
    "거래금액_만원"
]].copy()

# 2. MariaDB 테이블에 맞게 컬럼명 변경
apt_sql.columns = [
    "contract_month",
    "district",
    "location",
    "apartment_name",
    "area",
    "floor",
    "build_year",
    "price_manwon"
]

# 3. 날짜와 숫자 데이터 타입 변환
apt_sql["contract_month"] = pd.to_datetime(
    apt_sql["contract_month"]
).dt.date

for col in ["area", "floor", "build_year", "price_manwon"]:
    apt_sql[col] = pd.to_numeric(apt_sql[col], errors="coerce")

# 변환 과정에서 발생한 결측치 확인
if apt_sql.isna().any().any():
    print(apt_sql.isna().sum())
    raise ValueError("결측치가 있어 저장을 중단합니다.")

# 4. 데이터베이스 연결
conn = get_conn("realestate_project")
cur = conn.cursor()

try:
    # 기존 데이터가 있는지 확인
    cur.execute("SELECT COUNT(*) FROM apartment_sales")
    existing_count = cur.fetchone()[0]

    if existing_count > 0:
        print("이미 저장된 데이터:", existing_count, "건")
        print("중복 저장 방지를 위해 작업을 중단합니다.")

    else:
        # 저장할 데이터를 튜플로 변환
        rows = list(apt_sql.itertuples(index=False, name=None))

        sql = """
        INSERT INTO apartment_sales
        (contract_month, district, location, apartment_name,
         area, floor, build_year, price_manwon)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """

        # 1000건씩 나누어 저장
        for i in range(0, len(rows), 1000):
            cur.executemany(sql, rows[i:i+1000])

        conn.commit()

        # 저장 결과 확인
        cur.execute("SELECT COUNT(*) FROM apartment_sales")
        print("저장 완료:", cur.fetchone()[0], "건")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()

이미 저장된 데이터: 113083 건
중복 저장 방지를 위해 작업을 중단합니다.


In [50]:
# 데이터베이스 건수와 전처리 결과 비교
conn = get_conn("realestate_project")
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM apartment_sales")
        db_count = cur.fetchone()[0]
finally:
    conn.close()

print("Python 정제 건수:", len(apt_clean))
print("MariaDB 저장 건수:", db_count)
assert db_count == len(apt_clean), "Python과 MariaDB 거래 건수가 일치하지 않습니다."
print("정합성 검증 완료")

Python 정제 건수: 113083
MariaDB 저장 건수: 113083
정합성 검증 완료


## 6. SQL 통계 분석

연도별, 월별, 자치구별 거래건수와 평균 거래가격을 SQL로 계산합니다.

In [51]:
# SQL 조회 결과를 데이터프레임으로 가져오는 함수

def query_to_dataframe(sql):
    conn = get_conn("realestate_project")
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql)
            rows = cursor.fetchall()
            columns = [col[0] for col in cursor.description]
        return pd.DataFrame(rows, columns=columns)
    finally:
        conn.close()

In [52]:
# 연도별 거래 건수 및 평균가격
sql_yearly = """
SELECT
    YEAR(contract_month) AS 거래연도,
    COUNT(*) AS 거래건수,
    ROUND(AVG(price_manwon), 1) AS 평균가격_만원
FROM apartment_sales
GROUP BY YEAR(contract_month)
ORDER BY 거래연도
"""

yearly_sql_stats = query_to_dataframe(sql_yearly)
display(yearly_sql_stats)

,거래연도,거래건수,평균가격_만원
0,2020,25169,92853.0
1,2021,11888,115812.5
2,2022,2719,121809.4
3,2023,10768,116916.7
4,2024,18507,127722.4
5,2025,26105,136709.4
6,2026,17927,129262.7


In [53]:
# 월별 거래 건수 및 평균가격
sql_monthly = """
SELECT
    DATE_FORMAT(contract_month, '%Y-%m') AS 계약월,
    COUNT(*) AS 거래건수,
    ROUND(AVG(price_manwon), 1) AS 평균가격_만원
FROM apartment_sales
GROUP BY DATE_FORMAT(contract_month, '%Y-%m')
ORDER BY 계약월
"""

monthly_sql_stats = query_to_dataframe(sql_monthly)
print("월별 행 수:", len(monthly_sql_stats))
display(monthly_sql_stats.head())
display(monthly_sql_stats.tail())

월별 행 수: 83


,계약월,거래건수,평균가격_만원
0,2020-01,2017,73440.9
1,2020-02,2616,77600.7
2,2020-03,1321,74904.4
3,2020-04,850,77526.7
4,2020-05,1650,88371.4


,계약월,거래건수,평균가격_만원
78,2026-07,1761,129996.1
79,2026-08,1124,122548.9
80,2026-09,1024,130380.0
81,2026-10,1221,136711.2
82,2026-11,296,120827.4


In [54]:
# 2025년 자치구별 거래 건수 및 평균가격
sql_district_2025 = """
SELECT
    district AS 자치구,
    COUNT(*) AS 거래건수,
    ROUND(AVG(price_manwon) / 10000, 2) AS 평균가격_억원
FROM apartment_sales
WHERE contract_month >= '2025-01-01'
  AND contract_month < '2026-01-01'
GROUP BY district
ORDER BY 평균가격_억원 DESC
"""

district_2025_sql = query_to_dataframe(sql_district_2025)
print("2025년 거래 건수:", district_2025_sql["거래건수"].sum())
display(district_2025_sql)

2025년 거래 건수: 26105


,자치구,거래건수,평균가격_억원
0,서초구,892,27.88
1,강남구,996,26.30
2,송파구,1899,19.92
3,용산구,341,18.98
4,성동구,1601,16.88
5,마포구,1529,16.13
6,광진구,798,15.71
7,종로구,208,15.54
8,동작구,1779,14.16
9,강동구,1958,13.74


In [55]:
# Python과 SQL의 거래 건수 비교
python_month = monthly_stats[["계약월", "거래건수"]].copy()
python_month["계약월"] = python_month["계약월"].dt.strftime("%Y-%m")

month_check = python_month.merge(
    monthly_sql_stats[["계약월", "거래건수"]],
    on="계약월", how="outer", suffixes=("_python", "_sql"),
    validate="one_to_one"
)
assert month_check["거래건수_python"].equals(month_check["거래건수_sql"]), "월별 거래건수가 일치하지 않습니다."

python_district = district_stats[["자치구", "거래건수"]]
district_check = python_district.merge(
    district_2025_sql[["자치구", "거래건수"]],
    on="자치구", how="outer", suffixes=("_python", "_sql"),
    validate="one_to_one"
)
assert district_check["거래건수_python"].equals(district_check["거래건수_sql"]), "자치구별 거래건수가 일치하지 않습니다."

print("월별 거래건수 및 2025년 자치구별 거래건수 교차 검증 완료")

월별 거래건수 및 2025년 자치구별 거래건수 교차 검증 완료


## 7. 결과 정리

CSV 7개에서 총 153,428건을 불러왔고, 전용면적 83~85㎡에 해당하는 118,707건을 추출했습니다. 여기서 계약 해제 거래 5,624건을 제외한 최종 분석 대상은 113,083건입니다.

중복 의심 거래는 509건, 상세 정보가 동일한 거래는 349건이었습니다. IQR 기준 이상치 후보는 5,420건이었지만, 모두 원자료에 남겼습니다.

2025년 자치구별 거래건수와 평균·중위 거래가격을 비교했습니다. 2026년 자료는 11월 20일까지 수집한 결과여서 다른 연도와 직접 비교할 때 주의가 필요합니다.